# Experiment 4: CGM + time_since_meal + image (vit, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+vit (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'vit'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_vit'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_vit_stability_summary.json'

In [4]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=768, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_vit' / f'gated_cgm_time_vit_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_vit_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM+time+vit (gated) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.53 | MAE 9.06 | gRMSE 15.70 | Zone A 94.91% | A+B 99.58%
 30 min | RMSE 20.25 | MAE 13.25 | gRMSE 24.38 | Zone A 89.06% | A+B 99.35%
 60 min | RMSE 28.86 | MAE 18.71 | gRMSE 35.93 | Zone A 80.43% | A+B 98.70%
 90 min | RMSE 34.50 | MAE 22.44 | gRMSE 43.70 | Zone A 74.25% | A+B 97.91%
120 min | RMSE 37.73 | MAE 24.84 | gRMSE 48.26 | Zone A 70.47% | A+B 97.18%

--------------------------------------------------------------------------------
Test results — CGM+time+vit (gated) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 14.26 | MAE 10.35 | gRMSE 15.45 | Zone A 93.69% | A+B 99.60%
 30 min | RMSE 20.09 | MAE 13.75 | gRMSE 22.97 | Zone A 87.83% | A+B 99.48%
 60 min | RMSE 28.69 | MAE 19.31 | gRMSE 34.77 | Zone A 78.65% | A+B 98.92%
 90 min | RMSE 34.17 | MAE 23.10 | gRMSE 42.34 | Zone A 72.46% | A+B 98.35%
120 min | RMSE 37.30 | MAE 25.52 | gRMSE 46.75 | Zone A 67.84% | A+B 98.00%

--------------------------------------------------------------------------------
Test results — CGM+time+vit (gated) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


...

 15 min | RMSE 13.80 | MAE 9.18 | gRMSE 16.14 | Zone A 95.08% | A+B 99.60%
 30 min | RMSE 20.35 | MAE 13.28 | gRMSE 24.60 | Zone A 89.17% | A+B 99.38%
 60 min | RMSE 28.83 | MAE 18.62 | gRMSE 35.91 | Zone A 80.47% | A+B 98.72%
 90 min | RMSE 34.37 | MAE 22.27 | gRMSE 43.57 | Zone A 74.65% | A+B 97.95%
120 min | RMSE 37.41 | MAE 24.48 | gRMSE 47.85 | Zone A 71.26% | A+B 97.26%

--------------------------------------------------------------------------------
Test results — CGM+time+vit (gated) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.31 | MAE 8.79 | gRMSE 15.01 | Zone A 95.02% | A+B 99.63%
 30 min | RMSE 19.91 | MAE 12.97 | gRMSE 23.52 | Zone A 89.17% | A+B 99.42%
 60 min | RMSE 28.75 | MAE 18.53 | gRMSE 35.64 | Zone A 80.69% | A+B 98.74%
 90 min | RMSE 34.72 | MAE 22.41 | gRMSE 44.10 | Zone A 74.49% | A+B 97.86%
120 min | RMSE 37.90 | MAE 24.79 | gRMSE 48.57 | Zone A 70.77% | A+B 97.07%

--------------------------------------------------------------------------------
Test results — CGM+time+vit (gated) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.34 | MAE 8.84 | gRMSE 15.33 | Zone A 95.37% | A+B 99.64%
 30 min | RMSE 19.92 | MAE 12.96 | gRMSE 23.71 | Zone A 89.77% | A+B 99.43%
 60 min | RMSE 28.81 | MAE 18.51 | gRMSE 35.78 | Zone A 81.07% | A+B 98.69%
 90 min | RMSE 34.41 | MAE 22.12 | gRMSE 43.57 | Zone A 75.39% | A+B 97.97%
120 min | RMSE 37.62 | MAE 24.53 | gRMSE 48.05 | Zone A 71.49% | A+B 97.29%

--------------------------------------------------------------------------------


In [7]:
# Test Results:
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in all_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in all_results]
        else:
            seed_values = [r['test_results'][h][m] for r in all_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in all_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS, 'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== CGM+time+vit (gated) test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 13.63 +/- 0.36 (2.47) mg/dL   MAE = 9.17 +/- 0.49 (1.83)   gRMSE = 15.56 +/- 0.52 (4.27)   Zone A = 94.96 +/- 0.48 (3.29)%   Zone A+B = 99.61 +/- 0.02 (0.69)%
 30-min:  RMSE = 20.10 +/- 0.26 (4.73) mg/dL   MAE = 13.20 +/- 0.29 (3.12)   gRMSE = 23.86 +/- 0.61 (8.08)   Zone A = 89.15 +/- 0.51 (4.25)%   Zone A+B = 99.41 +/- 0.04 (0.76)%
 60-min:  RMSE = 28.84 +/- 0.14 (10.10) mg/dL   MAE = 18.65 +/- 0.25 (6.06)   gRMSE = 35.77 +/- 0.43 (16.14)   Zone A = 80.56 +/- 0.77 (5.40)%   Zone A+B = 98.72 +/- 0.08 (1.32)%
 90-min:  RMSE = 34.49 +/- 0.21 (14.04) mg/dL   MAE = 22.38 +/- 0.29 (8.20)   gRMSE = 43.62 +/- 0.57 (22.06)   Zone A = 74.56 +/- 0.91 (7.64)%   Zone A+B = 97.96 +/- 0.18 (2.45)%
120-min:  RMSE = 37.62 +/- 0.25 (16.50) mg/dL   MAE = 24.70 +/- 0.32 (9.72)   gRMSE = 48.01 +/- 0.60 (25.80)   Zone A = 70.77 +/- 1.09 (9.02)%   Zone A+B = 97.32 +/- 0.31 (3.83)%

saved per-seed results to /c